### 목표
1. 표준화 이해
2. 왜 KNN에서 표준화가 중요한지
3. 표준화 전/후 정확도 비교
4. fit_transform과 transform 차이
5. Pipeline 사용 이유 이해

---

# 1. 전처리에서 표준화란?
전처리: 원본 데이터를 모델이 학습하기 좋은 형태로 바꾸는 모든 작업

표준화는 feature 값을 평균 0, 표준편차 1 근처로 맞추는 것

```text
표준화 전:
feature 마다 값 범위가 제각각

표준화 후:
각 feature가 비슷한 스케일을 가짐
```

공식은 아래와 같다.
- z = (z - 평균) / 표준편차

EX:
어떤 feature가 1000, 2000, 3000 단위,
다른 feature가 0.1, 0.2, 0.3 단위
- KNN은 큰 숫자 feature에 더 끌려간다.
- KNN은 거리 기반 모델이라서 표준화 영향이 크다

---

# 2. 데이터 불러오기

In [20]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import pandas as pd

cancer = load_breast_cancer()

X = cancer.data
y = cancer.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print()

print("클래스 이름:", cancer.target_names)
print("feature 개수:", len(cancer.feature_names))

X shape: (569, 30)
y shape: (569,)

클래스 이름: ['malignant' 'benign']
feature 개수: 30


X = 종양 관련 측정값 30개

y = 악성 / 양성

---

# 3. feature값 범위 확인

표준화가 왜 필요한지 보기 위헤 feature들의 값 범위 확인 시

In [21]:
df = pd.DataFrame(X, columns=cancer.feature_names)  # columns=cancer.feature_names  는 각 열의 의미(이름)를 보여줌
df.describe().T[                    # .T 는 transpose, 즉 행과 열을 뒤집는다 => 각 feature 를 한 줄씩 보기좋게 바꿈
    ["min", "max", "mean", "std"]   # 전체 통계량 중에 4개만 선택함
].head(10)                          # 위에서부터 10개 행만 보여줌
# 유방암 데이터셋 X를 DataFrame으로 바꾼 뒤, 각 feature의 기초 통계량 일부만 뽑아서 위에서 10개 feature만 보는 코드

,min,max,mean,std
mean radius,6.98100,28.11000,14.127292,3.524049
mean texture,9.71000,39.28000,19.289649,4.301036
mean perimeter,43.79000,188.50000,91.969033,24.298981
mean area,143.50000,2501.00000,654.889104,351.914129
mean smoothness,0.05263,0.16340,0.096360,0.014064
mean compactness,0.01938,0.34540,0.104341,0.052813
mean concavity,0.00000,0.42680,0.088799,0.079720
mean concave points,0.00000,0.20120,0.048919,0.038803
mean symmetry,0.10600,0.30400,0.181162,0.027414
mean fractal dimension,0.04996,0.09744,0.062798,0.007060


예를 들어, 어떤 feature 는 값이 0.0 ~ 0.4

어떤 feature 는 100~4000 단위일 수도 있음

그러면 KNN에서 거리 계산시, 더 큰 단위 feature가 결과를 지배함

- 값 범위 큰 feature -> 거리 계산에 큰 영향
- 값 범위 작은 feature -> 상대적으로 영향 작아짐

---

# 4. Train/Test 분리

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # y(정답)의 클래스 비율을 유지하면서 나누라는 의미
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (455, 30)
X_test: (114, 30)
y_train: (455,)
y_test: (114,)


여기서 stratify=y 는 클래스 비율을 유지하면서 나누라는 의미이다.

쉽게말해서, y에 들어있는 정답 라벨의 비율을 train set 과 test set 에 비슷하게 나누겠다는 뜻이다.

EX:
    예를 들어 전체 데이터 y가 이렇게 구성되어 있다고 하면
- 전체 데이터

    - 0번 클래스: 40%
    - 1번 클래스: 60%

그러면 train_test_split()을 할 때 그냥 랜덤으로 나누면 우연히 이렇게 될 수도 있다.

- X_train
    - 0번 클래스: 35%
    - 1번 클래스: 65%

- X_test
    - 0번 클래스: 55%
    - 1번 클래스: 45%

이러면 train 데이터와 test 데이터의 클래스 분포가 달라져서 모델 평가가 왜곡될 수 있다.

그런데 stratify=y 를 넣으면 최대한 아래처럼 나눠준다.

- X_train
    - 0번 클래스: 40%
    - 1번 클래스: 60%

- X_test
    - 0번 클래스: 40%
    - 1번 클래스: 60%

즉, 전체 데이터의 정답 비율을 훈련 데이터와 테스트 데이터에도 비슷하게 유지하는 것이다.




분류 문제에서는 습관적으로 넣어주는 게 좋다.

- 전체 데이터에서 악성/양성 비율

    ≈ train 데이터에서 악성/양성 비율

    ≈ test 데이터에서 악성/양성 비율

---

# 5. 표준화 없이 KNN 실행

일단 표준화를 하지 않고 KNN 실행 시

In [23]:
from sklearn.neighbors import KNeighborsClassifier

# KNN 모델 생성
model = KNeighborsClassifier()

# 학습
model.fit(X_train, y_train)

# 평가
train_score_no_scaled = model.score(X_train, y_train)
test_score_no_scaled = model.score(X_test, y_test)

print("표준화 없음")
print("Train 정확도:", train_score_no_scaled)
print("Test 정확도:", test_score_no_scaled)

표준화 없음
Train 정확도: 0.9472527472527472
Test 정확도: 0.9122807017543859


---

# 6. StandardScaler 로 표준화하기

표준화 적용 시

In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# train 데이터 기준으로 평균, 표준편차 계산 후 변환
X_train_scaled = scaler.fit_transform(X_train)

# test 데이터는 train에서 구한 기준으로만 변환 
X_test_scaled = scaler.transform(X_test)

### 여기서 제일 중요한 것은 
- X_train -> fit_transform()
- X_test  -> transform()

절대 이렇게 하면 안된다.
- X_test_scaled = scalar.fit_transform(X_test)

왜냐하면, test 데이터의 평균과 표준편차를 따로 계산하면, 테스트 데이터 정보가 전처리에 반영된다.

### 이걸 데이터 누수라고 한다.

---

# 7. 표준화 후 값 확인

표준화가 제대로 됐는지 확인해보면

In [25]:
scaled_df = pd.DataFrame(X_train_scaled, columns=cancer.feature_names)
scaled_df.describe().T[
    ["mean", "std"]
].head(10)

,mean,std
mean radius,-4.337434e-15,1.001101
mean texture,2.240942e-15,1.001101
mean perimeter,-7.437274e-16,1.001101
mean area,1.503071e-16,1.001101
mean smoothness,5.223660e-15,1.001101
mean compactness,-2.775802e-15,1.001101
mean concavity,-7.046866e-16,1.001101
mean concave points,6.031805e-16,1.001101
mean symmetry,-3.263812e-15,1.001101
mean fractal dimension,-3.031519e-15,1.001101


이상적으로는 
- 평균 = 0
- 분산 = 1
그런데 완전히 동일하지 않아도 된다.

소수점 오차가 있을 수 있다.

---

# 8. 표준화 후 KNN 실행

In [26]:
model_scaled = KNeighborsClassifier(n_neighbors=3)

model_scaled.fit(X_train_scaled, y_train)

train_score_scaled = model_scaled.score(X_train_scaled, y_train)
test_score_scaled = model_scaled.score(X_test_scaled, y_test)

print("표준화 적용")
print("Train 정확도:", train_score_scaled)
print("Test 정확도:", test_score_scaled)

표준화 적용
Train 정확도: 0.978021978021978
Test 정확도: 0.9824561403508771


---

# 9. 표준화 전/후 비교표 만들기

In [27]:
compare = pd.DataFrame({
    "방법": ["표준화 없음", "표준화 적용"],
    "Train 정확도": [train_score_no_scaled, train_score_scaled],
    "Test 정확도": [test_score_no_scaled, test_score_scaled]
})

compare

,방법,Train 정확도,Test 정확도
0,표준화 없음,0.947253,0.912281
1,표준화 적용,0.978022,0.982456


위 표를 보면 표준화 후 성능이 좋아졌음을 확인할 수 있다.

---

# 10. Pipeline 으로 더 깔끔하게 쓰기

실제로는 아래 방식이 더 권장된다.

Pipeline 사용시 개발자가 fit_transform, transform 나눠줄 필요가 없다.

In [28]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

model_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=3))
])

model_pipeline.fit(X_train, y_train)

train_score_pipeline = model_pipeline.score(X_train, y_train)
test_score_pipeline = model_pipeline.score(X_test, y_test)

print("Pipeline 사용")
print("Train 정확도", train_score_pipeline)
print("Test 정확도:", test_score_pipeline)

Pipeline 사용
Train 정확도 0.978021978021978
Test 정확도: 0.9824561403508771


### Pipeline은 아래의 과정을 자동으로 묶어준다.

- 표준화
- KNN 학습
- 예측
- 평가

장점:
실수를 줄여준다.

GridSearchCV, cross_val_score 할 때는 Pipeline을 쓰는 게 좋다

---

# 11. 전체 코드

In [30]:
from sklearn.datasets import load_breast_cancer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd

# 1. 데이터 불러오기
cancer  = load_breast_cancer()
X = cancer.data
y = cancer.target

# 2. train_test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. 표준화 없는 KNN
model_no_scaled = KNeighborsClassifier(n_neighbors=3)
model_no_scaled.fit(X_train, y_train)

train_no_scaled = model_no_scaled.score(X_train, y_train)
test_no_scaled = model_no_scaled.score(X_test, y_test)

# 4. 표준화 적용 KNN
model_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=3))
])

model_scaled.fit(X_train, y_train)

train_scaled = model_scaled.score(X_train, y_train)
test_scaled = model_scaled.score(X_test, y_test)

# 5. 결과 비교
compare = pd.DataFrame({
    "방법": ["표준화 없음", "표준화 적용"],
    "train 정확도": [train_no_scaled, test_no_scaled],
    "test 정확도": [train_scaled, test_scaled]
})
compare

,방법,train 정확도,test 정확도
0,표준화 없음,0.953846,0.978022
1,표준화 적용,0.929825,0.982456
